In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model, optimizers, losses

2026-03-01 23:50:20.671449: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-01 23:50:21.990310: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2026-03-01 23:50:21.990450: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory
2026-03-01 23:50:21.990459: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Cannot dlopen some TensorRT libraries. If you would like to use Nv

In [3]:
# Hyperparameters
batch_size = 16
block_size = 32  # Context length
max_epochs = 25
learning_rate = 1e-3
n_embd = 32
n_head = 2
n_layer = 3
dropout = 0.0

In [4]:
# Step 1: Load the data (hardcoded conversational snippet, repeated)
text = """First Citizen:
Before we proceed any further, hear you this.

Second Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.
""" * 20  # ~12k characters

In [5]:
# Step 2: Cleaning the data (build char-level tokenizer)
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [6]:
# Convert text to numpy array (TF uses numpy for datasets)
data = np.array(encode(text), dtype=np.int32)

# Step 3: Split the data
n = int(0.9 * len(data))  # 90% train, 10% val
train_data = data[:n]
val_data = data[n:]

# Dataset generator
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = np.random.randint(len(data) - block_size, size=batch_size)
    x = np.stack([data[i:i+block_size] for i in ix])
    y = np.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

# Custom Transformer Block
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim)]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs, use_causal_mask=True)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [7]:
# GPT Model
class GPT(Model):
    def __init__(self):
        super(GPT, self).__init__()
        self.token_embedding = layers.Embedding(vocab_size, n_embd)
        self.position_embedding = layers.Embedding(block_size, n_embd)
        self.blocks = [TransformerBlock(n_embd, n_head, n_embd * 4, dropout) for _ in range(n_layer)]
        self.ln_f = layers.LayerNormalization(epsilon=1e-6)
        self.head = layers.Dense(vocab_size)

    def call(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(tf.range(T))
        x = tok_emb + pos_emb
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        if targets is None:
            return logits
        else:
            loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
            return logits, loss(targets, logits)

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits = self(idx_cond)
            logits = logits[:, -1, :]
            probs = tf.nn.softmax(logits, axis=-1)
            idx_next = tf.random.categorical(tf.math.log(probs), num_samples=1)
            idx = tf.concat([idx, idx_next], axis=1)
        return idx

In [9]:
# Initialize model and optimizer
model = GPT()
optimizer = optimizers.Adam(learning_rate=learning_rate)

# Compile for training (custom loop for loss)
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits, loss_value = model(x, y)
    grads = tape.gradient(loss_value, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss_value

In [10]:
# Step 4: Train the model
train_losses = []
for epoch in range(max_epochs):
    xb, yb = get_batch('train')
    loss = train_step(xb, yb)
    train_losses.append(loss.numpy())
    if epoch % 100 == 0 or epoch == max_epochs - 1:
        print(f"Epoch {epoch}: Training loss {loss.numpy():.4f}")

Epoch 0: Training loss 4.2410
Epoch 24: Training loss 2.8655


In [11]:
# Step 5: Evaluate the performance
def estimate_loss(split, eval_iters=20):
    losses = []
    for _ in range(eval_iters):
        x, y = get_batch(split)
        _, loss = model(x, y)
        losses.append(loss.numpy())
    return np.mean(losses)

In [12]:
train_loss = estimate_loss('train')
val_loss = estimate_loss('val')
print(f"Final train loss: {train_loss:.4f}")
print(f"Final val loss: {val_loss:.4f}")
perplexity = np.exp(val_loss)
print(f"Perplexity: {perplexity:.4f}")

Final train loss: 2.8895
Final val loss: 2.8977
Perplexity: 18.1323


In [13]:
# Generate text examples (single-turn chatbot)
prompt = "User: What is the price of corn?\nBot:\n"
context = np.array([encode(prompt)], dtype=np.int32)
generated = model.generate(tf.constant(context), max_new_tokens=100)
generated_text = prompt + decode(generated.numpy()[0, len(encode(prompt)):].tolist())
print("Generated conversational text:")
print(generated_text)

KeyError: 'U'